# End-to-End PySpark Data Engineering Project


## Task 1: Data Ingestion & Exploration


In [2]:
# 1. Install Dependencies
!pip install  pyspark findspark openml pyarrow

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.5 MB/s eta 0:00:0000:01
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=d2a71f04e1a944aaa07fddd652c2e219cf105aaa2b1065683abc2b1eb252dbcc
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [3]:
#SparkSession: This is the universal entry point for programming Spark with the Dataset and DataFrame AP
from pyspark.sql import SparkSession # type: ignore
#These are specialized, highly optimized functions that execute on distributed Spark data nodes 
from pyspark.sql.functions import * # type: ignore
#from pyspark.sql.window import Window
from pyspark.sql.window import Window # type: ignore
from pyspark.sql.types import * # type: ignore
import pandas as pd # type: ignore
import statsmodels.api as sm # type: ignore


spark=SparkSession.builder\
     .appName("PySpark_Data_Engineering_Project") \
          .master("local[*]") \
              .config("spark.driver.memory", "4g") \
                  .getOrCreate()

## Google Drive Mount

In [4]:
from google.colab import drive # type: ignore
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print("Loading dataset from drive...")
df = spark.read.csv(
    "/content/drive/MyDrive/BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)

Loading dataset from drive...


In [6]:
# Display Dataset Overview
print('Schema')
df.printSchema()


Schema
root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



### Count

In [7]:
print(f"Record Count: {df.count()}") #type:ignore

Record Count: 1000


In [8]:
print("Summary Statistics:")
df.describe().show()

Summary Statistics:
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|summary|      customer_id|               age|     tenure_months|  monthly_charges|     total_charges| contract_type|internet_service|   support_tickets|payment_method|             churn|
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|  count|             1000|              1000|              1000|             1000|              1000|          1000|            1000|              1000|          1000|              1000|
|   mean|            500.5|            43.819|            35.459|79.96715000000002| 2800.235379999997|          NULL|            NULL|             1.956|          NULL|             0.502|
| stddev|288.8194360957494|14.9910296500

###  Null Count

In [9]:
df.select([count(when(col(c).isNull(),c)).alias(c)for c in df.columns]).show() #type:ignore

+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



### Duplicate Count

In [10]:
duplicates = df.count() - df.dropDuplicates().count()
print(f"Duplicate Count: {duplicates}")

Duplicate Count: 0


### Data Types

In [11]:
print("Data Types :")
df.dtypes

Data Types :


[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

# Production-Ready Data Pipeline (Bronze Layer)

In [ ]:
import os

# Define the root path in your Google Drive where you want the project data to live
base_path = "/content/drive/MyDrive/pyspark_openml_project/data"

# Create the folders inside Google Drive
for layer in ['bronze', 'silver', 'gold']:
    folder = f"{base_path}/{layer}"
    os.makedirs(folder, exist_ok=True)
    
# Write to Bronze layer in Google Drive
df.write.mode("overwrite").parquet(f"{base_path}/bronze/raw_data.parquet")
print("Saved to Bronze layer in Google Drive.")


Saved to Bronze layer in Google Drive.


## Task 2: ETL Pipeline Development


In [16]:
# Extract from Bronze
df_bronze = spark.read.parquet(f"{base_path}/bronze/raw_data.parquet")

In [17]:
df_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



In [18]:
# 1. Duplicate Removal
df_transformed = df_bronze.dropDuplicates()

In [21]:
# 2. Missing Value Treatment
num_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets']
df_transformed = df_transformed.fillna(0.0, subset=num_cols)
df_transformed.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [ ]:
# 3. Data Type Conversion
df_transformed = df_transformed.withColumn("churn", col("churn").cast("int")) \
                               .withColumn("total_charges", col("total_charges").cast("double")) \
                               .withColumn("age", col("age").cast("integer"))

In [23]:
# 4. Feature Engineering
# Create a new feature 'is_senior' based on age
df_transformed = df_transformed.withColumn("is_senior", when(col("age") >= 60, 1).otherwise(0))
# Create 'charge_per_tenure' to see how much they pay relative to their tenure
df_transformed = df_transformed.withColumn("charge_per_tenure", col("total_charges") / (col("tenure_months") + 1))

In [24]:
# 5. Aggregation
# Calculate average monthly charges by contract type and join it back to the main DataFrame
contract_avg_df = df_transformed.groupBy("contract_type").agg(avg("monthly_charges").alias("avg_contract_monthly_charges"))
df_transformed = df_transformed.join(broadcast(contract_avg_df), on="contract_type", how="left")

In [25]:
df_transformed.show()

+--------------+-----------+---+-------------+---------------+-------------+----------------+---------------+--------------+-----+---------+------------------+----------------------------+
| contract_type|customer_id|age|tenure_months|monthly_charges|total_charges|internet_service|support_tickets|payment_method|churn|is_senior| charge_per_tenure|avg_contract_monthly_charges|
+--------------+-----------+---+-------------+---------------+-------------+----------------+---------------+--------------+-----+---------+------------------+----------------------------+
|Month-to-Month|          1| 56|           15|          59.23|       929.62|           Fiber|              5|           UPI|    1|        0|          58.10125|           80.42109777015438|
|Month-to-Month|        182| 23|           38|          80.86|      3246.77|             DSL|              3|          Cash|    1|        0| 83.25051282051282|           80.42109777015438|
|      Two Year|        199| 20|           40|         

# Load Store transformed data in:
## silver_layer/



In [29]:
df_transformed.write.mode("overwrite").parquet(f"{base_path}/silver/cleaned_data.parquet")
print("Saved to Silver layer. Here is a preview of the transformed data:")

Saved to Silver layer. Here is a preview of the transformed data:


## Task 3: ELT Pipeline & Medallion Architecture (Gold Layer)


In [30]:
# Read from Silver
df_silver = spark.read.parquet(f"{base_path}/silver/cleaned_data.parquet")

In [36]:
df_silver.show()

+--------------+-----------+---+-------------+---------------+-------------+----------------+---------------+--------------+-----+---------+------------------+----------------------------+
| contract_type|customer_id|age|tenure_months|monthly_charges|total_charges|internet_service|support_tickets|payment_method|churn|is_senior| charge_per_tenure|avg_contract_monthly_charges|
+--------------+-----------+---+-------------+---------------+-------------+----------------+---------------+--------------+-----+---------+------------------+----------------------------+
|Month-to-Month|          1| 56|           15|          59.23|       929.62|           Fiber|              5|           UPI|    1|        0|          58.10125|           80.42109777015438|
|Month-to-Month|        182| 23|           38|          80.86|      3246.77|             DSL|              3|          Cash|    1|        0| 83.25051282051282|           80.42109777015438|
|      Two Year|        199| 20|           40|         

In [34]:
#Generate minimum 3 business KPIs.
#1. Churn Distribution
kpi1_churn_dist = df_silver.groupBy("churn").count().withColumnRenamed("count", "total_customers")
kpi1_churn_dist.show()

+-----+---------------+
|churn|total_customers|
+-----+---------------+
|    1|            502|
|    0|            498|
+-----+---------------+



In [35]:
# 2. Average Monthly Charges by Churn
kpi2_avg_charges = df_silver.groupBy("churn").agg(avg("monthly_charges").alias("avg_monthly_charges"))
kpi2_avg_charges.show()

+-----+-------------------+
|churn|avg_monthly_charges|
+-----+-------------------+
|    1|  86.39589641434263|
|    0|  73.48676706827304|
+-----+-------------------+



In [38]:
# 3. Churn by Contract Type
kpi3_contract_churn = df_silver.groupBy("contract_type", "churn").count()
kpi3_contract_churn.show()

+--------------+-----+-----+
| contract_type|churn|count|
+--------------+-----+-----+
|      Two Year|    1|   30|
|      One Year|    0|  211|
|Month-to-Month|    1|  405|
|      Two Year|    0|  109|
|Month-to-Month|    0|  178|
|      One Year|    1|   67|
+--------------+-----+-----+



In [40]:
# Save to Gold
kpi1_churn_dist.write.mode("overwrite").parquet(f"{base_path}/gold/kpi1_churn_dist.parquet")
kpi2_avg_charges.write.mode("overwrite").parquet(f"{base_path}/gold/kpi2_avg_charges.parquet")
kpi3_contract_churn.write.mode("overwrite").parquet(f"{base_path}/gold/kpi3_contract_churn.parquet")
print("Saved KPIs to Gold layer.")

Saved KPIs to Gold layer.


## Task 4: PySpark + Pandas Integration (with statsmodels feature engineering)


In [ ]:
# Convert Spark to Pandas